# 05 - Pipeline ETL y Modelo Dimensional Kimball

**Proyecto:** Solución Analítica de Datos – Hotel Dann Monasterio  
**Metodología:** CRISP-DM / Kimball Dimensional Modeling  
**Objetivo específico:** OE2-A — Diseñar y construir el repositorio de datos dimensional  

Este notebook implementa el pipeline **ETL en 7 pasos**, construye el **esquema en estrella**
(Fact_Reservas + 6 dimensiones) y exporta el **DDL SQL** para MySQL Workbench.

### Jerarquía financiera del dataset
```
valorplan + ivaplan + servicioplan  =  totalconsumosplan
totalconsumosplan + totalconsumosadicional  =  ingreso_total (métrica central)
```

### Contenido
1. Extracción — carga del dataset limpio (parquet)
2. Estandarización de tipos y fechas
3. Variables derivadas (incluye ingreso_total con fórmula correcta)
4. Anonimización PII (SHA-256)
5. Construcción de 6 dimensiones
6. Construcción de Fact_Reservas
7. Exportación CSV + validación del modelo


## Librerías e importaciones

In [ ]:
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Librerías cargadas')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

## Paso 1 — Extracción del dataset limpio

Cargamos el dataset depurado generado por `02_limpieza_datos.ipynb`.
Si el parquet no existe, cargamos el Excel fuente como fallback.


In [ ]:
PARQUET_PATH = Path('../data/processed/reservas_clean.parquet')
RAW_PATH     = Path('../data/raw/DataSet_ReservaYHuespedes_V.Full.xlsx')
OUT_DIR      = Path('../data/processed/kimball')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    print(f'Cargado desde parquet: {df.shape}')
else:
    print('Parquet no encontrado. Cargando desde Excel...')
    h1 = pd.read_excel(RAW_PATH, sheet_name='Hoja1')
    h2 = pd.read_excel(RAW_PATH, sheet_name='Hoja2')
    df_raw = pd.concat([h1, h2], ignore_index=True)
    print(f'Excel cargado: {df_raw.shape}')
    nulos_pct = df_raw.isnull().mean() * 100
    cols_drop = nulos_pct[nulos_pct >= 95].index.tolist()
    df = df_raw.drop(columns=cols_drop).copy()
    print(f'Columnas eliminadas (>=95% nulos): {len(cols_drop)}')

print(f'Shape final: {df.shape}')
df.dtypes.value_counts()

## Paso 2 — Estandarización de tipos y fechas

Convertimos columnas de fecha a `datetime64[ns]` y columnas financieras a `float64`.


In [ ]:
cols_fecha = [c for c in ['fllega_aco','fsalid_aco','fcheckout','fechasischin'] if c in df.columns]
for col in cols_fecha:
    df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

cols_num = [c for c in ['tarifa','valorplan','ivaplan','servicioplan',
                         'valorconsumoadicional','totalconsumosadicional',
                         'totalconsumosplan','edad_aco'] if c in df.columns]
for col in cols_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Tipos estandarizados')
df[cols_fecha + [c for c in cols_num if c in df.columns]].dtypes

## Paso 3 — Variables derivadas

### Fórmula de ingreso (jerarquía financiera del dataset)

```
valorplan + ivaplan + servicioplan  =  totalconsumosplan
totalconsumosplan + totalconsumosadicional  =  ingreso_total
```

Si `totalconsumosplan` ya está calculado en el dataset limpio, se usa directamente.
En caso contrario, se reconstruye sumando sus tres componentes.


In [ ]:
# --- ingreso_total (fórmula correcta) -----------------------------------
if 'totalconsumosplan' in df.columns:
    # totalconsumosplan ya incluye valorplan + ivaplan + servicioplan
    df['ingreso_total'] = (
        df['totalconsumosplan'].fillna(0) +
        df['totalconsumosadicional'].fillna(0)
    ).clip(lower=0)
    print('ingreso_total = totalconsumosplan + totalconsumosadicional')
else:
    # Fallback: reconstruir totalconsumosplan desde sus componentes
    col_val  = df['valorplan'].fillna(0)     if 'valorplan'   in df.columns else 0
    col_iva  = df['ivaplan'].fillna(0)       if 'ivaplan'     in df.columns else 0
    col_srv  = df['servicioplan'].fillna(0)  if 'servicioplan' in df.columns else 0
    col_adic = df['totalconsumosadicional'].fillna(0) if 'totalconsumosadicional' in df.columns else 0
    df['totalconsumosplan_calc'] = (col_val + col_iva + col_srv).clip(lower=0)
    df['ingreso_total']          = (df['totalconsumosplan_calc'] + col_adic).clip(lower=0)
    print('ingreso_total = (valorplan+ivaplan+servicioplan) + totalconsumosadicional')

# --- Duración de estancia -----------------------------------------------
if 'fllega_aco' in df.columns and 'fsalid_aco' in df.columns:
    df['duracion_estancia'] = (df['fsalid_aco'] - df['fllega_aco']).dt.days
    df.loc[df['duracion_estancia'] < 0,  'duracion_estancia'] = np.nan
    df.loc[df['duracion_estancia'] > 60, 'duracion_estancia'] = np.nan

# --- Lead time -----------------------------------------------------------
if 'fechasischin' in df.columns and 'fllega_aco' in df.columns:
    df['lead_time'] = (df['fllega_aco'] - df['fechasischin']).dt.days
    df.loc[df['lead_time'] < 0,   'lead_time'] = 0
    df.loc[df['lead_time'] > 365, 'lead_time'] = np.nan

# --- Componentes de fecha ------------------------------------------------
if 'fllega_aco' in df.columns:
    df['anio']       = df['fllega_aco'].dt.year
    df['mes']        = df['fllega_aco'].dt.month
    df['trimestre']  = df['fllega_aco'].dt.quarter
    df['dia_semana'] = df['fllega_aco'].dt.day_name()

print()
print(df[['ingreso_total','duracion_estancia','lead_time']].describe().round(2))

## Paso 4 — Anonimización de datos PII

Hash SHA-256 (truncado a 12 hex) sobre `ident_aco` → `id_huesped`.


In [ ]:
def hash_id(valor):
    if pd.isna(valor):
        return 'ANONIMO'
    return hashlib.sha256(str(valor).encode()).hexdigest()[:12].upper()

if 'ident_aco' in df.columns:
    df['id_huesped'] = df['ident_aco'].apply(hash_id)
    print(f'id_huesped generado | unicos: {df["id_huesped"].nunique():,}')
else:
    df['id_huesped'] = 'ANONIMO'
    print('ident_aco no encontrado — id_huesped = ANONIMO')

## Paso 5 — Construcción del Modelo Dimensional (Kimball)

### 5.1 — Dim_Fecha

In [ ]:
fechas_validas = df['fllega_aco'].dropna()
rango = pd.date_range(start=fechas_validas.min().normalize(),
                      end=fechas_validas.max().normalize(), freq='D')

dim_fecha = pd.DataFrame({'fecha': rango})
dim_fecha['id_fecha']       = dim_fecha['fecha'].dt.strftime('%Y%m%d').astype(int)
dim_fecha['anio']           = dim_fecha['fecha'].dt.year
dim_fecha['mes']            = dim_fecha['fecha'].dt.month
dim_fecha['nombre_mes']     = dim_fecha['fecha'].dt.strftime('%b').str.upper()
dim_fecha['trimestre']      = dim_fecha['fecha'].dt.quarter
dim_fecha['semana_anio']    = dim_fecha['fecha'].dt.isocalendar().week.astype(int)
dim_fecha['dia_semana_num'] = dim_fecha['fecha'].dt.dayofweek + 1
dim_fecha['dia_semana']     = dim_fecha['fecha'].dt.day_name()
dim_fecha['es_fin_semana']  = (dim_fecha['dia_semana_num'] >= 6).astype(int)
dim_fecha['semestre']       = dim_fecha['mes'].apply(lambda m: 1 if m <= 6 else 2)
dim_fecha = dim_fecha[['id_fecha','fecha','anio','semestre','trimestre','mes',
                        'nombre_mes','semana_anio','dia_semana_num','dia_semana','es_fin_semana']]
print(f'Dim_Fecha: {dim_fecha.shape}  |  {dim_fecha["anio"].min()} - {dim_fecha["anio"].max()}')
dim_fecha.head(3)

### 5.2 — Dim_Segmento

In [ ]:
mapa_seg = {
    'COR': ('Corporativo',          'Empresas con contrato corporativo regular',               'Corporativo'),
    'CE':  ('Corporativo Especial', 'Empresas con tarifas preferenciales negociadas',           'Corporativo'),
    'ME':  ('Mostrador/Externo',    'Reservas walk-in o directas sin convenio',                 'Transiente'),
    'EM':  ('Empleados',            'Reservas para empleados del hotel o convenios laborales',  'Interno'),
    'T&T': ('Tour & Travel',        'Agencias de viaje y operadores turisticos',                'Agencias'),
}
segs = df['codsegmento'].dropna().unique() if 'codsegmento' in df.columns else []
rows = []
for i, s in enumerate(sorted(segs), start=1):
    nombre, desc, tipo = mapa_seg.get(s, (s, 'Sin clasificar', 'Otro'))
    rows.append({'id_segmento': i, 'codigo_segmento': s,
                 'nombre_segmento': nombre, 'descripcion': desc, 'tipo_cliente': tipo})
dim_segmento = pd.DataFrame(rows)
print(f'Dim_Segmento: {dim_segmento.shape}')
print(dim_segmento.to_string(index=False))

### 5.3 — Dim_Canal

In [ ]:
mapa_tipo_canal = {
    'BKNG':'OTA', 'HDAN':'Directo Digital', 'RECE':'Directo Presencial',
    'CORP':'Corporativo', 'AGCY':'Agencia', 'EXPE':'OTA',
}
if 'codiga_age' in df.columns and 'nombre_age' in df.columns:
    canales = (df[['codiga_age','nombre_age']]
               .dropna(subset=['codiga_age']).drop_duplicates(subset=['codiga_age'])
               .sort_values('codiga_age').reset_index(drop=True))
    canales.insert(0, 'id_canal', range(1, len(canales)+1))
    canales.columns = ['id_canal','codigo_canal','nombre_canal']
    canales['tipo_canal'] = canales['codigo_canal'].map(mapa_tipo_canal).fillna('Otro')
    canales['es_online']  = canales['tipo_canal'].isin(['OTA','Directo Digital']).astype(int)
    dim_canal = canales
    print(f'Dim_Canal: {dim_canal.shape}')
    print(dim_canal.head(10).to_string(index=False))

### 5.4 — Dim_Habitacion

In [ ]:
mapa_hab = {
    'S3':('Suite Junior',   2,'Suite'),   'SE':('Suite Estandar',   2,'Suite'),
    'ST':('Suite Superior', 2,'Suite'),   'SG':('Sencilla',         1,'Estandar'),
    'DB':('Doble',          2,'Estandar'),'CD':('Cuadruple',        4,'Estandar'),
}
if 'tiphab_tip' in df.columns and 'clahab_clh' in df.columns:
    combo = (df[['tiphab_tip','clahab_clh']].dropna(subset=['tiphab_tip'])
             .drop_duplicates().sort_values(['tiphab_tip','clahab_clh']).reset_index(drop=True))
    combo.insert(0,'id_habitacion', range(1, len(combo)+1))
    combo.columns = ['id_habitacion','tipo_hab','clase_hab']
    combo['descripcion_tipo'] = combo['tipo_hab'].map(lambda x: mapa_hab.get(x,(x,'',1))[0])
    combo['capacidad_max']    = combo['tipo_hab'].map(lambda x: mapa_hab.get(x,(x,'',1))[1])
    combo['categoria']        = combo['tipo_hab'].map(lambda x: mapa_hab.get(x,(x,'','Otro'))[2])
    dim_habitacion = combo
    print(f'Dim_Habitacion: {dim_habitacion.shape}')
    print(dim_habitacion.to_string(index=False))

### 5.5 — Dim_Huesped

In [ ]:
if 'rango_edad' not in df.columns and 'edad_aco' in df.columns:
    bins   = [0, 25, 35, 50, 65, 120]
    labels = ['18-25','26-35','36-50','51-65','65+']
    df['rango_edad'] = pd.cut(df['edad_aco'], bins=bins, labels=labels, right=True)

cols_h = [c for c in ['id_huesped','sexo_aco','rango_edad','nacionalidad',
                       'oficio','nombre_emp'] if c in df.columns]
dim_huesped = (df[cols_h].drop_duplicates(subset=['id_huesped'])
               .reset_index(drop=True).copy())
dim_huesped.insert(0, 'id_registro_huesped', range(1, len(dim_huesped)+1))
if 'sexo_aco' in dim_huesped.columns:
    dim_huesped['sexo_aco'] = dim_huesped['sexo_aco'].str.upper().map(
        {'M':'Masculino','F':'Femenino','MASCULINO':'Masculino','FEMENINO':'Femenino'}
    ).fillna('No especificado')
print(f'Dim_Huesped: {dim_huesped.shape}')
dim_huesped.head(4)

### 5.6 — Dim_Temporada

In [ ]:
mapa_temp = {
    'A':('Alta', 'Temporada alta: dic-ene, julio, festivos'),
    'B':('Baja', 'Temporada baja: meses intermedios sin festivos'),
    'M':('Media','Temporada intermedia'),
}
if 'codigotemporada' in df.columns:
    temps = df['codigotemporada'].dropna().unique()
    rows = []
    for i, t in enumerate(sorted(temps), start=1):
        nombre_t, desc_t = mapa_temp.get(t.strip().upper(), (t, 'Sin descripcion'))
        nombre_db = ''
        if 'nombretemporada' in df.columns:
            vals = df.loc[df['codigotemporada']==t, 'nombretemporada'].dropna()
            nombre_db = vals.iloc[0] if len(vals) else ''
        rows.append({'id_temporada':i,'codigo_temporada':t,'nombre_temporada':nombre_t,
                     'descripcion':desc_t,'nombre_en_sistema':nombre_db})
    rows.append({'id_temporada':99,'codigo_temporada':'ND','nombre_temporada':'No Disponible',
                 'descripcion':'Sin temporada registrada (53.27% de registros — limitacion documentada)',
                 'nombre_en_sistema':'NULL'})
    dim_temporada = pd.DataFrame(rows)
    print(f'Dim_Temporada: {dim_temporada.shape}')
    print(dim_temporada.to_string(index=False))

## Paso 6 — Construcción de Fact_Reservas

La tabla de hechos conecta todas las dimensiones mediante llaves foráneas
y contiene las métricas cuantificables de cada reserva.


In [ ]:
# Llaves dimensionales
if 'fllega_aco' in df.columns:
    df['id_fecha'] = pd.to_numeric(
        df['fllega_aco'].dt.strftime('%Y%m%d'), errors='coerce').fillna(0).astype(int)

if 'codsegmento' in df.columns:
    seg_map = dim_segmento.set_index('codigo_segmento')['id_segmento'].to_dict()
    df['id_segmento'] = df['codsegmento'].map(seg_map).fillna(0).astype(int)

if 'codiga_age' in df.columns:
    can_map = dim_canal.set_index('codigo_canal')['id_canal'].to_dict()
    df['id_canal'] = df['codiga_age'].map(can_map).fillna(0).astype(int)

if 'tiphab_tip' in df.columns:
    hab_map = dim_habitacion.set_index('tipo_hab')['id_habitacion'].to_dict()
    df['id_habitacion'] = df['tiphab_tip'].map(hab_map).fillna(0).astype(int)

if 'codigotemporada' in df.columns:
    temp_map = dim_temporada.set_index('codigo_temporada')['id_temporada'].to_dict()
    df['id_temporada'] = df['codigotemporada'].map(temp_map).fillna(99).astype(int)

print('Llaves dimensionales generadas')

In [ ]:
# Construir Fact_Reservas
cols_fact = ['id_fecha','id_segmento','id_canal','id_habitacion','id_temporada']
if 'id_huesped' in df.columns:
    cols_fact.append('id_huesped')

medidas = ['numvoucher','ingreso_total','tarifa','valorplan','ivaplan','servicioplan',
           'valorconsumoadicional','totalconsumosadicional','totalconsumosplan',
           'duracion_estancia','lead_time']
cols_fact += [c for c in medidas if c in df.columns]

fact_reservas = df[cols_fact].copy()
fact_reservas.insert(0, 'id_reserva', range(1, len(fact_reservas)+1))

# Redondear métricas monetarias
for col in ['ingreso_total','tarifa','valorplan','ivaplan','servicioplan',
            'valorconsumoadicional','totalconsumosadicional','totalconsumosplan']:
    if col in fact_reservas.columns:
        fact_reservas[col] = fact_reservas[col].round(2)

print(f'Fact_Reservas: {fact_reservas.shape}')
fact_reservas.head(4)

## Paso 7 — Exportación de tablas a CSV y validación del modelo

Exportamos cada tabla a `data/processed/kimball/` para importar en MySQL Workbench.
Orden de carga: Dims primero → Fact_Reservas al final.


In [ ]:
tablas = {
    'Dim_Fecha':      dim_fecha,      'Dim_Segmento':  dim_segmento,
    'Dim_Canal':      dim_canal,      'Dim_Habitacion':dim_habitacion,
    'Dim_Huesped':    dim_huesped,    'Dim_Temporada': dim_temporada,
    'Fact_Reservas':  fact_reservas,
}
for nombre, tabla in tablas.items():
    ruta = OUT_DIR / f'{nombre}.csv'
    tabla.to_csv(ruta, index=False, encoding='utf-8-sig')
    print(f'{nombre}.csv  ->  {tabla.shape[0]:,} filas x {tabla.shape[1]} cols')

In [ ]:
# Validación de integridad referencial
print('=' * 55)
print('VALIDACION DE INTEGRIDAD REFERENCIAL')
print('=' * 55)
checks = [
    ('id_fecha',      dim_fecha,      'id_fecha'),
    ('id_segmento',   dim_segmento,   'id_segmento'),
    ('id_canal',      dim_canal,      'id_canal'),
    ('id_habitacion', dim_habitacion, 'id_habitacion'),
    ('id_temporada',  dim_temporada,  'id_temporada'),
]
for fk, dim, pk in checks:
    if fk not in fact_reservas.columns:
        print(f'  -- {fk}: no encontrado')
        continue
    fact_vals = set(fact_reservas[fk].dropna().unique())
    dim_vals  = set(dim[pk].unique())
    orphans   = fact_vals - dim_vals
    pct       = (1 - len(orphans)/max(len(fact_vals),1)) * 100
    estado    = 'OK' if len(orphans) == 0 else 'REVISAR'
    print(f'  [{estado}] {fk}: {pct:.1f}% cobertura | {len(orphans)} huerfanos')

print()
print(f'Registros Fact_Reservas : {len(fact_reservas):,}')
print(f'Ingreso total del periodo: COP {fact_reservas["ingreso_total"].sum():,.0f}')

## Resumen del esquema en estrella


In [ ]:
print('Tabla                     Filas      Cols')
print('-' * 42)
for nombre, tabla in tablas.items():
    print(f'{nombre:<25}  {len(tabla):>8,}  {tabla.shape[1]:>5}')
print()
print('CSVs exportados a: ../data/processed/kimball/')
print('DDL SQL disponible en: ../sql/01_ddl_kimball.sql')

# Conclusiones del notebook 05

**Pipeline ETL — 7 pasos ejecutados:**
1. Extracción desde parquet (o fallback Excel 2 hojas, 70,882 registros)
2. Estandarización de tipos y fechas
3. Variables derivadas con fórmula correcta de ingreso:
   `totalconsumosplan + totalconsumosadicional = ingreso_total`
4. Anonimización SHA-256 de datos PII (campo `ident_aco`)
5. Construcción de 6 tablas dimensión
6. Construcción de Fact_Reservas con FKs + métricas monetarias y temporales
7. Exportación CSV + validación de integridad referencial (>=95%)

**Modelo dimensional generado:**
- `Dim_Fecha` — calendario Jun 2020 – Abr 2026
- `Dim_Segmento` — 5 segmentos (COR, CE, ME, EM, T&T)
- `Dim_Canal` — agencias/canales con clasificación online/offline
- `Dim_Habitacion` — 6 tipos con categoría y capacidad máxima
- `Dim_Huesped` — perfil demográfico anonimizado
- `Dim_Temporada` — temporadas + registro ND para nulos documentados
- `Fact_Reservas` — tabla central con FKs + 10 métricas

**Siguiente paso:** ejecutar `01_ddl_kimball.sql` en MySQL Workbench
e importar los CSVs de `data/processed/kimball/`.
